# ISOM 835 · Session 2 — Data Wrangling & EDA for Prediction
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Sep 21 · Prof. Hasan Arslan**

Real data arrives in several tables, three date formats, and a thousand blanks. Tonight: pandas joins and groupbys, datetime features, missing-value patterns, EDA that asks questions — and the anatomy of **target leakage**.

> **Frame the prediction (Olist).** *Unit:* one order · *Target:* will the customer leave a low review (1–2 stars)? · *Horizon:* at delivery · *Decision:* proactive customer-service outreach before the review is written.

Two data paths in this notebook:
1. **Olist Brazilian e-commerce** (Kaggle, ~100k orders in 8 tables) — needs a free Kaggle account; loaded with `kagglehub`.
2. **No-login fallback** — the Telco churn CSV from the course repo, used for every join/groupby idea when Kaggle isn't available.

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 40)
print('pandas', pd.__version__)

## 1. pandas in 2026 — two things that changed
pandas **3.0** (January 2026) made *Copy-on-Write* the only mode and made a real string dtype the default. The practical consequence: **chained assignment no longer writes**. Write it the safe way every time.

In [ ]:
df = pd.DataFrame({'a': [1, 2, 3], 'b': [10, 20, 30]})
# ❌ old idiom — may silently do nothing under Copy-on-Write:
#    df[df['a'] > 1]['b'] = 0
# ✅ always use .loc with a single indexing step:
df.loc[df['a'] > 1, 'b'] = 0
df

In [ ]:
# OPTIONAL — the other dataframe library. Polars (1.x) is 5–20× faster on big joins/groupbys and scikit-learn can output Polars frames.
try:
    import polars as pl
    tpl = pl.read_csv('https://raw.githubusercontent.com/harslan/isom-835/master/public/data/telco_churn.csv')
    print(tpl.group_by('Contract').agg((pl.col('Churn') == 'Yes').mean().alias('churn_rate'), pl.len().alias('n')).sort('churn_rate', descending=True))
except ImportError:
    print('polars not installed — pip install polars (pandas stays the course default)')

## 2. Load the data
Run the Kaggle cell if you have an account (Colab will prompt for your token the first time). Otherwise the fallback cell loads Telco and we proceed with the same techniques.

In [ ]:
# OPTIONAL — Kaggle path (needs a free Kaggle account). Skip if it errors; the next cell is the fallback.
import kagglehub, os
path = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
files = {f.replace('olist_', '').replace('_dataset.csv', ''): os.path.join(path, f) for f in os.listdir(path) if f.endswith('.csv')}
orders    = pd.read_csv(files['orders'], parse_dates=[c for c in ['order_purchase_timestamp','order_delivered_customer_date','order_estimated_delivery_date'] ])
items     = pd.read_csv(files['order_items'])
reviews   = pd.read_csv(files['order_reviews'])
customers = pd.read_csv(files['customers'])
print({k: pd.read_csv(v, nrows=1).shape[1] for k, v in files.items()})
print(orders.shape, items.shape, reviews.shape, customers.shape)
HAVE_OLIST = True

In [ ]:
# Fallback: Telco churn from the course repo (no login). Runs whether or not the Kaggle cell worked.
URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/telco_churn.csv'
telco = pd.read_csv(URL)
HAVE_OLIST = 'HAVE_OLIST' in dir() and HAVE_OLIST
print('Telco:', telco.shape, '| Olist available:', HAVE_OLIST)
telco.head(3)

## 3. Know your table before you model it
Shape, dtypes, blanks, and the one column that is secretly a string. `TotalCharges` in Telco has 11 blank strings — pandas reads the whole column as text.

In [ ]:
print(telco.dtypes.value_counts())
telco['TotalCharges'] = pd.to_numeric(telco['TotalCharges'], errors='coerce')   # blanks → NaN
print('missing TotalCharges:', telco['TotalCharges'].isna().sum())
telco[telco['TotalCharges'].isna()][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head()

Those 11 customers all have `tenure == 0` — brand-new accounts that have never been billed. The blank is not random; it *means* something. That is the difference between **MCAR** (missing completely at random), **MAR** (missing depends on other columns — here, tenure), and **MNAR** (missing depends on the missing value itself, e.g. high earners skipping the income question).

## 4. Joins — where row counts go wrong
A one-to-many merge silently multiplies rows. **Always check the row count before and after.** With Olist: one order has many items and (usually) one review. With the Telco fallback we simulate a second table.

In [ ]:
if HAVE_OLIST:
    n0 = len(orders)
    m = orders.merge(items, on='order_id', how='left')            # one order → many items
    print(f'orders {n0:,} → after item join {len(m):,}  (rows multiplied: {len(m)/n0:.2f}x)')
    # collapse items back to one row per order BEFORE joining to reviews
    per_order = items.groupby('order_id').agg(n_items=('order_item_id', 'max'), revenue=('price', 'sum'), freight=('freight_value', 'sum')).reset_index()
    base = orders.merge(per_order, on='order_id', how='left').merge(reviews[['order_id', 'review_score']].drop_duplicates('order_id'), on='order_id', how='left')
    print(f'base table: {base.shape}  (should still be {n0:,} rows)')
else:
    # simulate a payments table with 1–3 rows per customer to see the multiplication
    rng = np.random.default_rng(835)
    pays = pd.DataFrame({'customerID': np.repeat(telco['customerID'].values, rng.integers(1, 4, len(telco)))})
    pays['amount'] = rng.normal(70, 20, len(pays)).round(2)
    n0 = len(telco); m = telco.merge(pays, on='customerID', how='left')
    print(f'customers {n0:,} → after payment join {len(m):,}  (rows multiplied: {len(m)/n0:.2f}x)')
    per_cust = pays.groupby('customerID').agg(n_payments=('amount', 'size'), total_paid=('amount', 'sum')).reset_index()
    base = telco.merge(per_cust, on='customerID', how='left')
    print(f'base table: {base.shape}  (should still be {n0:,} rows)')

**Rule:** aggregate the many-side to the grain of your prediction unit *first*, then join. If your target is per order, every feature must be per order.

## 5. Groupby — the EDA that asks questions
Every groupby is a hypothesis: *does X change the rate of Y?* If a bar chart shows no difference, no model will find one either.

In [ ]:
if HAVE_OLIST:
    base['low_review'] = (base['review_score'] <= 2).astype(int)
    base['late_days'] = (base['order_delivered_customer_date'] - base['order_estimated_delivery_date']).dt.days
    base['late_bucket'] = pd.cut(base['late_days'], [-999, -1, 0, 3, 7, 999], labels=['early', 'on time', '1-3 days late', '4-7 late', '8+ late'])
    rate = base.groupby('late_bucket', observed=True)['low_review'].agg(['mean', 'size'])
else:
    base['churn'] = (base['Churn'] == 'Yes').astype(int)
    base['tenure_bucket'] = pd.cut(base['tenure'], [-1, 6, 12, 24, 48, 72], labels=['0-6 mo', '7-12', '13-24', '25-48', '49-72'])
    rate = base.groupby('tenure_bucket', observed=True)['churn'].agg(['mean', 'size'])
print(rate.round(3))
rate['mean'].plot(kind='bar', color='#2ee6c5', figsize=(7, 3.5), title='Outcome rate by bucket'); plt.ylabel('rate'); plt.show()

Late delivery drives low reviews on Olist; short tenure drives churn on Telco. Either way the chart is the argument — and the feature you just built (`late_days`, `tenure_bucket`) is a hypothesis the model can test.

## 6. Datetime features
Dates are three features in disguise: **when** (hour, weekday, month), **how long** (durations between events), and **how recent** (days since). All must be computed with information available at prediction time.

In [ ]:
if HAVE_OLIST:
    base['purchase_dow'] = base['order_purchase_timestamp'].dt.dayofweek
    base['purchase_hour'] = base['order_purchase_timestamp'].dt.hour
    base['promised_days'] = (base['order_estimated_delivery_date'] - base['order_purchase_timestamp']).dt.days
    print(base.groupby('purchase_dow')['low_review'].mean().round(3))
else:
    # Telco has no dates — build a plausible signup date from tenure to practice the API
    asof = pd.Timestamp('2026-09-01')
    base['signup_date'] = asof - pd.to_timedelta(base['tenure'] * 30, unit='D')
    base['signup_month'] = base['signup_date'].dt.month
    base['days_as_customer'] = (asof - base['signup_date']).dt.days
    print(base[['tenure', 'signup_date', 'signup_month', 'days_as_customer']].head())

## 7. The anatomy of leakage
A feature **leaks** when it carries information that would not exist at the moment you need the prediction. Three real patterns:

| Pattern | Example | Why it leaks |
|---|---|---|
| Recorded after the outcome | Olist `review_score` → predicting low review; Bank Marketing `duration` | It *is* the outcome, or is known only once the outcome is |
| Downstream of the decision | `retention_offer_sent` → predicting churn | The business only sends offers to customers it already thinks will churn |
| Aggregated over the future | "customer's average review score" computed over all time | Includes reviews written after this order |

The test is always one question: **would I know this value at prediction time?**

In [ ]:
# A deliberately leaky feature, to see how good it looks — and why that's the warning sign
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

if HAVE_OLIST:
    d = base.dropna(subset=['low_review', 'late_days']).copy()
    y = d['low_review']
    honest = d[['promised_days', 'n_items', 'revenue', 'freight', 'purchase_dow']].fillna(0)
    leaky = honest.assign(late_days=d['late_days'])                     # known only AFTER delivery — leak if predicting at purchase
    note = 'late_days is fine if we predict at delivery, a leak if we predict at purchase'
else:
    d = base.copy(); y = d['churn']
    honest = pd.get_dummies(d[['tenure', 'MonthlyCharges', 'Contract']], drop_first=True)
    # fabricate a post-outcome field: 'account_closed_flag' (recorded when the customer actually leaves)
    leaky = honest.assign(account_closed_flag=(y == 1).astype(int) * (np.random.default_rng(1).random(len(y)) > 0.05))
    note = 'account_closed_flag is recorded after the churn happens — pure leakage'
for name, X in [('honest', honest), ('leaky', leaky)]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=835)
    p = LogisticRegression(max_iter=2000).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    print(f'{name:7s} AUC {roc_auc_score(yte, p):.3f}')
print(note)

An AUC that jumps to 0.95+ from one column is not a discovery. It is a leak — **"too good to be true" is a diagnostic.**

## 8. Your turn
1. **Missingness check.** For the Telco table, compute the churn rate for customers with `TotalCharges` missing vs. not. Is the blank MCAR, MAR, or MNAR?
2. **A hypothesis feature.** Build `charges_per_month = TotalCharges / tenure` (guard against tenure 0). Bucket it into quartiles and plot churn rate by quartile. Does it say something `MonthlyCharges` alone does not?
3. **Leak hunt.** For your own final-project idea, list every candidate feature and mark each one *known at prediction time* / *not known*. Post the list on Canvas with your HW1.

In [ ]:
# Your turn — work here

## What we learned tonight
- **Aggregate to the grain of the prediction unit, then join.** Check row counts before and after every merge.
- **Missingness carries information** — classify it (MCAR/MAR/MNAR) and, if it's informative, keep an indicator.
- **Every groupby is a hypothesis;** every datetime column is three features.
- **Leakage test:** would I know this at prediction time? A jump to a near-perfect score is a warning, not a win.

**Homework #1** (due Mon Sep 28): a full EDA + baseline on Telco — see the session page for the brief. Next week: encoders, scalers, imputers, and the Pipeline that makes leakage structurally impossible.